# 📊 SBI証券 AI投資判断システム

**使い方（3ステップ）**
1. 上メニュー → 「ランタイム」→「すべてのセルを実行」
2. セル3でSBIスクリーンショットをアップロード
3. 最後のセルにレポートが自動表示される

In [ ]:
import subprocess, sys
for pkg in ['easyocr', 'yfinance>=0.2.40', 'Pillow']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✅ パッケージ準備完了')

In [ ]:
import math, re, time
import yfinance as yf
from datetime import datetime, timedelta
from IPython.display import display, HTML

# ── テクニカル指標（scanner_v12継承） ──────────────────────────

def _ema(data, period):
    k = 2 / (period + 1)
    e = [data[0]]
    for v in data[1:]:
        e.append(v * k + e[-1] * (1 - k))
    return e

def _calc_rsi(closes, period=14):
    if len(closes) < period + 2: return 50.0
    gains, losses = [], []
    for i in range(1, len(closes)):
        d = closes[i] - closes[i-1]
        gains.append(max(d, 0.0)); losses.append(max(-d, 0.0))
    ag = sum(gains[:period]) / period
    al = sum(losses[:period]) / period
    for i in range(period, len(gains)):
        ag = (ag * (period-1) + gains[i]) / period
        al = (al * (period-1) + losses[i]) / period
    return 100.0 if al == 0 else 100 - 100 / (1 + ag / al)

def _calc_indicators(closes, highs, lows, volumes):
    if len(closes) < 80: return None
    ef = _ema(closes, 9); es = _ema(closes, 26)
    ml = [f - s for f, s in zip(ef, es)]
    sl = _ema(ml[25:], 6)
    hist_t = ml[-1] - sl[-1]; hist_p = ml[-2] - sl[-2]
    k14r = []
    for i in range(13, len(closes)):
        hi = max(highs[i-13:i+1]); lo = min(lows[i-13:i+1])
        k14r.append(50 if hi == lo else (closes[i]-lo)/(hi-lo)*100)
    sm14 = _ema(k14r, 3)
    ma25 = sum(closes[-25:])/25; ma25p = sum(closes[-26:-1])/25
    rsi  = _calc_rsi(closes[-30:])
    ma20 = sum(closes[-20:])/20
    std20 = math.sqrt(sum((c-ma20)**2 for c in closes[-20:])/20)
    bb_lo = ma20 - 2*std20; bb_hi = ma20 + 2*std20
    bb_pos = (closes[-1]-bb_lo)/(bb_hi-bb_lo) if bb_hi > bb_lo else 0.5
    vol_ma20 = sum(volumes[-20:])/20
    vol_ratio = volumes[-1]/vol_ma20 if vol_ma20 > 0 else 1.0
    ma5  = sum(closes[-5:])/5; ma5p = sum(closes[-6:-1])/5
    ma75 = sum(closes[-75:])/75
    return {
        'hist_t': hist_t, 'hist_chg': hist_t - hist_p,
        'k14': sm14[-1], 'k14_chg': sm14[-1] - sm14[-2],
        'dev': (closes[-1]-ma25)/ma25*100,
        'rsi': rsi, 'bb_pos': bb_pos, 'vol_ratio': vol_ratio,
        'ma5_slope': (ma5-ma5p)/ma5p*100,
        'dev75': (closes[-1]-ma75)/ma75*100,
        'price': closes[-1],
        'support': min(lows[-60:]), 'resistance': max(highs[-60:]),
        'price_change_5d': (closes[-1]-closes[-6])/closes[-6]*100 if len(closes)>=6 else 0,
        'price_change_20d': (closes[-1]-closes[-21])/closes[-21]*100 if len(closes)>=21 else 0,
    }

def fetch_ohlcv(code, days=200):
    """yfinance MultiIndex対応版"""
    ticker = f'{code}.T'
    end = datetime.now(); start = end - timedelta(days=days)
    try:
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if df is None or len(df) < 80: return None
        def _col(c):
            s = df[c]
            if hasattr(s, 'squeeze'): s = s.squeeze()
            return s.dropna().tolist()
        return {'closes': _col('Close'), 'highs': _col('High'),
                'lows': _col('Low'), 'volumes': _col('Volume')}
    except Exception as e:
        print(f'  [{code}] 取得失敗: {e}')
        return None

SECTOR_MAP = {
    '1605':('エネルギー','原油・天然ガス開発'),
    '1662':('エネルギー','石油資源開発'),
    '2782':('小売','100円ショップ'),
    '3901':('素材','難燃防護服・機能素材'),
    '4245':('環境','水処理・排水設備'),
    '4506':('医薬品','製薬（住友化学系）'),
    '4550':('医薬品','製薬（住友化学系）'),
    '3983':('素材','鉄鋼加工・金属資材'),
    '9602':('エンタメ','映画・不動産'),
    '9876':('小売','ファッション・アパレル'),
}

def judge_position(pos, ind):
    pnl = pos.get('pnl_pct') or 0
    code = pos['code']
    sector, detail = SECTOR_MAP.get(code, ('不明', ''))

    def tech_s():
        if not ind: return ''
        parts = []
        if ind['rsi'] <= 30:    parts.append(f"RSI{ind['rsi']:.0f}(売られ過ぎ)")
        elif ind['rsi'] >= 70:  parts.append(f"RSI{ind['rsi']:.0f}(買われ過ぎ)")
        else:                    parts.append(f"RSI{ind['rsi']:.0f}")
        if ind['bb_pos'] < 0.2:  parts.append('BB下限近傍')
        elif ind['bb_pos'] > 0.8: parts.append('BB上限近傍')
        if ind['hist_chg'] > 0:  parts.append('MACD改善')
        elif ind['hist_chg'] < 0: parts.append('MACD悪化')
        return '｜'.join(parts) + '。' if parts else ''

    if pnl <= -12:
        return ('即売却（損切）','★★★',
                f'損益率{pnl:.1f}%は信用取引の危機水域。追証リスク回避のため即座に損切りを実行せよ。{detail}の回復を待つ余裕はない。',
                'HIGH')
    if pnl <= -7:
        return ('即売却（損切）','★★★',
                f'損益率{pnl:.1f}%は損切りライン完全突破。テクニカル回復の証拠がない限り、損失拡大前の撤退が鉄則。',
                'HIGH')
    if pnl <= -5:
        if ind and ind['bb_pos'] < 0.15 and ind['rsi'] < 30 and ind['k14'] < 20:
            return ('保有継続（反発監視）','★★',
                    f'損益{pnl:.1f}%だがBB下限({ind["bb_pos"]:.2f}) / RSI{ind["rsi"]:.0f} / K14={ind["k14"]:.0f}で売られ過ぎ。{tech_s()}反発確認後に即利確前提で短期保有継続。',
                    'MEDIUM')
        return ('損切り','★★',
                f'損益{pnl:.1f}%。テクニカル反発根拠なし。{tech_s()}{detail}の業績悪化と複合すると損失拡大リスク大。',
                'MEDIUM')
    if sector == 'エネルギー' and pnl > 0:
        if ind and ind['rsi'] > 65:
            return ('一部利確検討','★★',
                    f'損益{pnl:+.1f}%。{tech_s()}RSI過熱圏。利益の50%を確定させ残りを保有するのが現実的。',
                    'LOW-MED')
        return ('保有継続','★★',
                f'損益{pnl:+.1f}%。{tech_s()}エネルギー安保テーマで中期支持継続。WTI60ドル割れが損切りトリガー。',
                'LOW')
    if ind and ind['k14'] <= 30 and ind['k14_chg'] >= 1 and ind['rsi'] <= 40 and ind['bb_pos'] <= 0.3 and pnl > -3:
        return ('追加買い候補','★★★',
                f'進化買いシグナル発動(77.8%勝率)。K14={ind["k14"]:.0f}(↑+{ind["k14_chg"]:.1f}pt) / RSI{ind["rsi"]:.0f} / BB下位({ind["bb_pos"]:.2f})。{tech_s()}エントリー強度高。',
                'LOW')
    if ind and ind['rsi'] >= 55 and ind['ma5_slope'] > 0.3 and ind['hist_chg'] > 0 and pnl > 0:
        return ('保有継続（上昇継続）','★★',
                f'損益{pnl:+.1f}%。{tech_s()}上昇モメンタム継続中。利益を伸ばす局面。',
                'LOW')
    if ind and ind['bb_pos'] < 0.2 and ind['rsi'] < 35 and pnl > -5:
        return ('保有継続（反発待ち）','★★',
                f'売られ過ぎ領域。{tech_s()}短期反発確率高。損益-5%到達で即損切り発動。',
                'LOW-MED')
    if -3 <= pnl <= 1:
        return ('保有継続（様子見）','★',
                f'損益{pnl:+.1f}%で方向感なし。{tech_s()}明確な上昇シグナルが出るまで現状維持。-5%達したら即損切り。',
                'LOW-MED')
    if pnl > 1:
        return ('保有継続','★★',f'損益{pnl:+.1f}%。{tech_s()}引き続き上昇余地を観察。','LOW')
    return ('損切り検討','★',f'損益{pnl:+.1f}%。{tech_s()}明確な反発根拠なし。','MEDIUM')

print('✅ エンジン定義完了')

In [ ]:
from google.colab import files
import easyocr, re
from PIL import Image

# ── 既知銘柄マスタ ────────────────────────────────────────────
KNOWN_NAMES = {
    '1605':'INPEX', '1662':'石油資源開発', '2782':'セリア',
    '3901':'アゼアス', '4245':'ダイキアクシス',
    '4550':'住友ファーマ', '4506':'住友ファーマ',
    '3983':'モリテック', '9602':'東宝', '9876':'コックス',
}

# ── Column-aware SBI OCRパーサー ─────────────────────────────
# SBIレイアウト: 行A=[名前][現在値][損益円]  行B=[コード][平均単価][損益率]
# コード行(B)を見つけ、直前行(A)をルックバックして価格を取得する

def _sf(s):
    """文字列 → float。失敗時 None"""
    try:
        return float(str(s).replace(',','').replace('，','')
                            .replace('−','-').replace('＋','+').replace('±',''))
    except: return None

def _get_prices(txt):
    RE = re.compile(r'(\d[\d,，]*(?:\.\d{1,2})?)(?:円|¥)?')
    return [v for p in RE.findall(txt) for v in [_sf(p)] if v and v > 50]

def _get_pct(txt):
    RE = re.compile(r'([+\-−]?\d{1,3}(?:\.\d{1,2})?)(?:%|％)')
    for p in RE.findall(txt):
        v = _sf(p)
        if v is not None and abs(v) < 100: return v
    return None

def _get_yen(txt):
    m = re.search(r'([+\-−±])\s*([\d,，]+(?:\.\d+)?)', txt)
    if m:
        sign = -1 if m.group(1) in ('-','−') else 1
        v = _sf(m.group(2))
        return sign * v if v is not None else None
    prices = _get_prices(txt)
    if not prices: return None
    return (-1 if re.search(r'[-−]', txt) else 1) * max(abs(p) for p in prices)

def parse_sbi_blocks(raw, img_w, img_h):
    RE_CODE = re.compile(r'\b(\d{4})\b')
    LEFT_MAX, RIGHT_MIN = 0.38, 0.65

    # カラム分類
    enriched = []
    for bbox, text, conf in raw:
        xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
        xn = sum(xs)/4 / img_w
        ya = sum(ys)/4
        col = 'L' if xn < LEFT_MAX else ('R' if xn > RIGHT_MIN else 'M')
        enriched.append({'t': text.strip(), 'ya': ya, 'xn': xn, 'col': col})

    enriched.sort(key=lambda b: (b['ya'], b['xn']))

    # 行グループ化
    tol = max(img_h * 0.025, 8)
    rows, cur, cy = [], [], None
    for b in enriched:
        if cy is None or abs(b['ya'] - cy) <= tol:
            cur.append(b); cy = b['ya'] if cy is None else (cy + b['ya'])/2
        else:
            rows.append(cur); cur, cy = [b], b['ya']
    if cur: rows.append(cur)

    # L/M/R に集約
    def lmr(row):
        L = ' '.join(b['t'] for b in row if b['col']=='L')
        M = ' '.join(b['t'] for b in row if b['col']=='M')
        R = ' '.join(b['t'] for b in row if b['col']=='R')
        return L.strip(), M.strip(), R.strip()
    collapsed = [lmr(r) for r in rows]

    # コード行 → ルックバックでペアリング
    positions = []
    for i, (L, M, R) in enumerate(collapsed):
        cm = RE_CODE.search(L)
        if not cm or not (1000 <= int(cm.group(1)) <= 9999): continue
        code = cm.group(1)
        name = KNOWN_NAMES.get(code, code)

        cur_price = avg_price = pnl_yen = None
        for j in range(i-1, max(i-4, -1), -1):
            pL, pM, pR = collapsed[j]
            prices = _get_prices(pM)
            if prices:
                cur_price = prices[0]
                pnl_yen   = _get_yen(pR)
                if code not in KNOWN_NAMES and pL.strip():
                    name = pL.strip()[:20]
                break

        avg_c = _get_prices(M)
        if avg_c: avg_price = avg_c[0]
        pnl_pct = _get_pct(R)

        if cur_price is None or avg_price is None: continue
        if pnl_pct is None:
            pnl_pct = (cur_price - avg_price) / avg_price * 100

        # 符号補正（OCRが - を落とすケース対策）
        if pnl_yen is not None:
            if pnl_pct < 0 and pnl_yen > 0: pnl_yen = -pnl_yen
            elif pnl_pct > 0 and pnl_yen < 0: pnl_yen = -pnl_yen

        positions.append({
            'code': code, 'name': name,
            'current_price': cur_price, 'avg_cost': avg_price,
            'pnl_pct': round(pnl_pct, 2),
            'pnl_yen': round(pnl_yen) if pnl_yen is not None else None,
        })
    return positions

# ── アップロード & OCR実行 ────────────────────────────────────
print('📸 SBIスクリーンショットをアップロードしてください...')
uploaded = files.upload()

if not uploaded:
    print('アップロードなし → 直近実績データ(2026/04/29)を使用')
    POSITIONS = [
        {'code':'1605','name':'INPEX',        'current_price':4123, 'avg_cost':4058.00,'pnl_pct':+1.52,'pnl_yen': None},
        {'code':'1662','name':'石油資源開発',  'current_price':2295, 'avg_cost':2269.70,'pnl_pct':+1.04,'pnl_yen': None},
        {'code':'2782','name':'セリア',        'current_price':3440, 'avg_cost':3555.00,'pnl_pct':-3.33,'pnl_yen':-11845},
        {'code':'3901','name':'アゼアス',      'current_price': 633, 'avg_cost': 684.00,'pnl_pct':-7.78,'pnl_yen':-30677},
        {'code':'4245','name':'ダイキアクシス','current_price': 703, 'avg_cost': 739.00,'pnl_pct':-5.13,'pnl_yen': -7596},
        {'code':'4506','name':'住友ファーマ',  'current_price':1851, 'avg_cost':2164.50,'pnl_pct':-14.80,'pnl_yen':-32145},
        {'code':'3983','name':'モリテック',    'current_price': 234, 'avg_cost': 232.86,'pnl_pct': -0.17,'pnl_yen':  -285},
        {'code':'9602','name':'東宝',          'current_price':1461, 'avg_cost':1465.00,'pnl_pct': -0.32,'pnl_yen':  -462},
        {'code':'9876','name':'コックス',      'current_price': 238, 'avg_cost': 270.00,'pnl_pct':-12.09,'pnl_yen': -9817},
    ]
else:
    img_path = list(uploaded.keys())[0]
    print(f'✅ アップロード完了: {img_path}')

    img = Image.open(img_path)
    img_w, img_h = img.size

    reader = easyocr.Reader(['ja','en'], gpu=False, verbose=False)
    raw = reader.readtext(img_path)
    print(f'OCR: {len(raw)} ブロック検出 (画像 {img_w}×{img_h}px)')

    POSITIONS = parse_sbi_blocks(raw, img_w, img_h)

    if not POSITIONS:
        print('⚠️ OCR解析失敗。サンプルデータを手動で編集して再実行してください。')

print(f'\n検出ポジション: {len(POSITIONS)}件')
for p in POSITIONS:
    cur = p['current_price']; avg = p['avg_cost']; pnl = p['pnl_pct'] or 0
    sign = '+' if pnl >= 0 else ''
    cur_s = f"{cur:>8,.1f}" if cur is not None else "    N/A "
    avg_s = f"{avg:>9,.2f}" if avg is not None else "      N/A"
    print(f"  {p['code']} {p['name']:12s}  現在値{cur_s}円  平均{avg_s}円  損益率{sign}{pnl:.2f}%")

In [ ]:
RESULTS = []
print(f'[分析] {len(POSITIONS)} 銘柄を取得中...\n')
for i, pos in enumerate(POSITIONS, 1):
    code = pos['code']; name = pos['name']
    print(f'  [{i}/{len(POSITIONS)}] {code} {name}', end=' ... ', flush=True)
    data = fetch_ohlcv(code)
    ind = None
    if data:
        try:
            ind = _calc_indicators(data['closes'], data['highs'], data['lows'], data['volumes'])
        except Exception as e:
            print(f'(指標計算エラー: {e})', end=' ')
    action, stars, reason, risk = judge_position(pos, ind)
    sector, detail = SECTOR_MAP.get(code, ('不明', ''))
    RESULTS.append({
        'code': code, 'name': name, 'sector': sector, 'detail': detail,
        'cur': pos.get('current_price') or 0,
        'avg': pos.get('avg_cost') or 0,
        'pnl_pct': pos.get('pnl_pct') or 0,
        'pnl_yen': pos.get('pnl_yen'),
        'action': action, 'stars': stars, 'reason': reason, 'risk': risk,
        'indicators': ind,
    })
    print(action)
    if i < len(POSITIONS): time.sleep(0.3)

print('\n✅ 分析完了')

In [ ]:
ORDER = {
    '即売却（損切）':0,'損切り':1,'損切り検討':2,'一部利確検討':3,
    '保有継続（反発監視）':4,'保有継続（反発待ち）':5,'追加買い候補':6,
    '保有継続（上昇継続）':7,'保有継続':8,'保有継続（様子見）':9,'様子見':10,
}
THEME = {
    '即売却（損切）':  {'bg':'#b71c1c','light':'#ffebee','text':'#b71c1c','icon':'🚨','label':'即損切り'},
    '損切り':          {'bg':'#c62828','light':'#ffebee','text':'#c62828','icon':'⛔','label':'損切り'},
    '損切り検討':      {'bg':'#e65100','light':'#fff3e0','text':'#e65100','icon':'⚠️','label':'損切り検討'},
    '一部利確検討':    {'bg':'#f57f17','light':'#fffde7','text':'#f57f17','icon':'💰','label':'一部利確'},
    '保有継続（反発監視）':{'bg':'#0d47a1','light':'#e3f2fd','text':'#0d47a1','icon':'👁','label':'反発監視'},
    '保有継続（反発待ち）':{'bg':'#1565c0','light':'#e3f2fd','text':'#1565c0','icon':'⏳','label':'反発待ち'},
    '追加買い候補':    {'bg':'#1b5e20','light':'#e8f5e9','text':'#1b5e20','icon':'🎯','label':'追加買い'},
    '保有継続（上昇継続）':{'bg':'#2e7d32','light':'#e8f5e9','text':'#2e7d32','icon':'🚀','label':'上昇継続'},
    '保有継続':        {'bg':'#388e3c','light':'#e8f5e9','text':'#388e3c','icon':'✅','label':'保有継続'},
    '保有継続（様子見）':  {'bg':'#546e7a','light':'#eceff1','text':'#546e7a','icon':'⏸','label':'様子見'},
    '様子見':          {'bg':'#607d8b','light':'#eceff1','text':'#607d8b','icon':'⏸','label':'様子見'},
}
RISK_STYLE = {
    'HIGH':    ('🔴','#b71c1c','#ffebee'),
    'MEDIUM':  ('🟠','#e65100','#fff3e0'),
    'LOW-MED': ('🟡','#f57f17','#fffde7'),
    'LOW':     ('🟢','#2e7d32','#e8f5e9'),
}

sorted_r = sorted(RESULTS, key=lambda r: ORDER.get(r['action'], 99))

total_yen = sum(r['pnl_yen'] for r in RESULTS if r.get('pnl_yen') is not None)
total_val = sum(r['avg'] for r in RESULTS if r.get('avg'))
total_pct = total_yen / total_val * 100 if total_val > 0 else 0

urgent_list = [r for r in RESULTS if r['action'] in ('即売却（損切）','損切り')]
caution_list= [r for r in RESULTS if '損切り検討' in r['action'] or '監視' in r['action']]
hold_list   = [r for r in RESULTS if '保有継続' in r['action']]
add_list    = [r for r in RESULTS if '追加買い' in r['action']]

def gauge_bar(val, lo, hi, warn_lo=None, warn_hi=None, width=100):
    pct = max(0, min(100, (val - lo) / (hi - lo) * 100))
    color = '#4caf50'
    if warn_lo is not None and val <= warn_lo: color = '#2196f3'
    if warn_hi is not None and val >= warn_hi: color = '#f44336'
    return (f"<div style='flex:1;background:#e0e0e0;border-radius:3px;height:5px;position:relative;min-width:40px'>"
            f"<div style='width:{pct:.0f}%;height:100%;background:{color};border-radius:3px'></div></div>")

def ind_block(ind):
    if not ind: return ''
    rsi_bar = gauge_bar(ind['rsi'], 0, 100, warn_lo=30, warn_hi=70)
    bb_bar  = gauge_bar(ind['bb_pos']*100, 0, 100, warn_lo=20, warn_hi=80)
    k14_bar = gauge_bar(ind['k14'], 0, 100, warn_lo=30, warn_hi=70)
    ch5 = ind.get('price_change_5d', 0)
    ch5_c = '#c62828' if ch5 > 0 else '#1565c0'
    return f"""
<div style='background:#f8f9fa;border-radius:8px;padding:10px 12px;margin:8px 0'>
  <div style='font-size:10px;font-weight:600;color:#9e9e9e;letter-spacing:.5px;margin-bottom:8px'>テクニカル指標</div>
  <div style='display:grid;grid-template-columns:1fr 1fr 1fr;gap:8px;margin-bottom:8px'>
    <div style='text-align:center'>
      <div style='font-size:10px;color:#757575;margin-bottom:2px'>RSI</div>
      <div style='font-size:16px;font-weight:700;color:{"#1565c0" if ind["rsi"]<=30 else "#c62828" if ind["rsi"]>=70 else "#212121"}'>{ind['rsi']:.0f}</div>
      {rsi_bar}
    </div>
    <div style='text-align:center'>
      <div style='font-size:10px;color:#757575;margin-bottom:2px'>BB位置</div>
      <div style='font-size:16px;font-weight:700;color:{"#1565c0" if ind["bb_pos"]<=0.2 else "#c62828" if ind["bb_pos"]>=0.8 else "#212121"}'>{ind['bb_pos']:.2f}</div>
      {bb_bar}
    </div>
    <div style='text-align:center'>
      <div style='font-size:10px;color:#757575;margin-bottom:2px'>K14</div>
      <div style='font-size:16px;font-weight:700;color:{"#1565c0" if ind["k14"]<=30 else "#c62828" if ind["k14"]>=70 else "#212121"}'>{ind['k14']:.0f}</div>
      {k14_bar}
    </div>
  </div>
  <div style='display:flex;gap:8px;flex-wrap:wrap'>
    <span style='font-size:11px;color:#546e7a;background:#fff;padding:2px 8px;border-radius:12px;border:1px solid #e0e0e0'>MA乖離 <b style='color:#212121'>{ind['dev']:+.1f}%</b></span>
    <span style='font-size:11px;color:#546e7a;background:#fff;padding:2px 8px;border-radius:12px;border:1px solid #e0e0e0'>出来高比 <b style='color:#212121'>{ind['vol_ratio']:.1f}x</b></span>
    <span style='font-size:11px;color:#546e7a;background:#fff;padding:2px 8px;border-radius:12px;border:1px solid #e0e0e0'>5D騰落 <b style='color:{ch5_c}'>{ch5:+.1f}%</b></span>
    <span style='font-size:11px;color:#546e7a;background:#fff;padding:2px 8px;border-radius:12px;border:1px solid #e0e0e0'>MACD <b style='color:{"#2e7d32" if ind["hist_chg"]>0 else "#c62828"}'>{"↑改善" if ind["hist_chg"]>0 else "↓悪化"}</b></span>
  </div>
</div>"""

cards_html = ''
for r in sorted_r:
    act = r['action']
    th  = THEME.get(act, THEME['様子見'])
    pnl = r['pnl_pct']
    pnl_c = '#c62828' if pnl > 0 else '#1565c0'
    sign  = '+' if pnl >= 0 else ''
    rk    = r['risk']
    rk_icon, rk_col, rk_bg = RISK_STYLE.get(rk, ('⚪','#9e9e9e','#f5f5f5'))
    ind_h = ind_block(r['indicators'])
    yen_str = f"評価損益 <b style='color:{pnl_c}'>{sign}{r['pnl_yen']:,}円</b> " if r.get('pnl_yen') is not None else ''

    cards_html += f"""
<div style='background:#fff;border-radius:14px;margin-bottom:12px;overflow:hidden;
            box-shadow:0 2px 8px rgba(0,0,0,.10);border-left:5px solid {th["bg"]}'>
  <div style='padding:14px 14px 0'>
    <div style='display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:10px'>
      <div>
        <span style='font-size:18px;font-weight:800;color:#1a1a2e;letter-spacing:.5px'>{r['code']}</span>
        <span style='font-size:14px;font-weight:500;color:#424242;margin-left:7px'>{r['name']}</span>
        <span style='display:inline-block;font-size:10px;background:#e8eaf6;color:#3949ab;
                     border-radius:10px;padding:2px 8px;margin-left:6px;font-weight:600'>{r['sector']}</span>
      </div>
      <div style='text-align:right'>
        <div style='font-size:22px;font-weight:800;color:{pnl_c};line-height:1'>{sign}{pnl:.2f}%</div>
        <div style='font-size:11px;color:#9e9e9e;margin-top:2px'>現在 {r['cur']:,.1f} / 平均 {r['avg']:,.2f}</div>
      </div>
    </div>
    <div style='display:flex;align-items:center;gap:8px;margin-bottom:10px;flex-wrap:wrap'>
      <span style='background:{th["bg"]};color:#fff;padding:6px 14px;border-radius:20px;
                   font-size:13px;font-weight:700;letter-spacing:.3px'>{th["icon"]} {act}</span>
      <span style='font-size:15px;color:#fbc02d;letter-spacing:2px'>{r['stars']}</span>
      <span style='background:{rk_bg};color:{rk_col};padding:3px 10px;border-radius:10px;
                   font-size:11px;font-weight:700;border:1px solid {rk_col}22'>{rk_icon} {rk}</span>
    </div>
    {f'<div style="font-size:12px;color:#757575;margin-bottom:6px">{yen_str}</div>' if yen_str else ''}
  </div>
  {ind_h and f'<div style="padding:0 14px">{ind_h}</div>'}
  <div style='padding:10px 14px 14px'>
    <div style='font-size:13px;line-height:1.7;color:#424242;background:{th["light"]};
                border-radius:8px;padding:10px 12px;border-left:3px solid {th["bg"]}'>{r['reason']}</div>
  </div>
</div>"""

# ── アラートバナー ──
alert_html = ''
if urgent_list:
    names = '、'.join(f"{r['name']}({r['code']})" for r in urgent_list)
    alert_html = f"""
<div style='background:#b71c1c;color:#fff;padding:12px 16px;border-radius:12px;margin-bottom:16px;
            display:flex;align-items:center;gap:10px'>
  <span style='font-size:20px'>🚨</span>
  <div>
    <div style='font-weight:700;font-size:14px'>即時対応が必要なポジション {len(urgent_list)}件</div>
    <div style='font-size:12px;opacity:.9;margin-top:2px'>{names}</div>
  </div>
</div>"""

# ── 総損益カード ──
tp_c = '#c62828' if total_yen > 0 else '#1565c0'
tp_sign = '+' if total_yen >= 0 else ''
pct_sign = '+' if total_pct >= 0 else ''

html_out = f"""<!DOCTYPE html>
<html lang="ja">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1,maximum-scale=1">
<style>
  *{{box-sizing:border-box;-webkit-tap-highlight-color:transparent}}
  body{{margin:0;padding:0;background:#f0f2f5;font-family:-apple-system,BlinkMacSystemFont,'Hiragino Sans',sans-serif}}
</style>
</head>
<body>
<div style='max-width:640px;margin:0 auto'>

  <!-- ヘッダー -->
  <div style='background:linear-gradient(135deg,#1a237e 0%,#283593 60%,#3949ab 100%);
              padding:20px 20px 16px;color:#fff'>
    <div style='display:flex;justify-content:space-between;align-items:center'>
      <div>
        <div style='font-size:11px;opacity:.7;letter-spacing:1px;text-transform:uppercase'>SBI証券</div>
        <div style='font-size:20px;font-weight:800;letter-spacing:.5px'>📊 投資判断レポート</div>
      </div>
      <div style='text-align:right;font-size:11px;opacity:.8'>{datetime.now().strftime('%Y/%m/%d')}<br>{datetime.now().strftime('%H:%M')}</div>
    </div>
  </div>

  <!-- ポートフォリオサマリー -->
  <div style='background:#fff;margin:12px 12px 0;border-radius:14px;padding:16px;
              box-shadow:0 2px 8px rgba(0,0,0,.10)'>
    <div style='font-size:11px;color:#9e9e9e;font-weight:600;letter-spacing:.5px;margin-bottom:6px'>評価損益合計（信用建玉）</div>
    <div style='font-size:32px;font-weight:800;color:{tp_c};line-height:1'>{tp_sign}{total_yen:,}円</div>
    <div style='font-size:15px;font-weight:600;color:{tp_c};margin-top:4px'>{pct_sign}{total_pct:.2f}%</div>
    <div style='display:flex;gap:8px;margin-top:14px;flex-wrap:wrap'>
      <div style='flex:1;min-width:70px;background:#ffebee;border-radius:10px;padding:8px 10px;text-align:center'>
        <div style='font-size:18px;font-weight:800;color:#b71c1c'>{len(urgent_list)}</div>
        <div style='font-size:10px;color:#c62828;font-weight:600;margin-top:2px'>🚨 即損切り</div>
      </div>
      <div style='flex:1;min-width:70px;background:#fff3e0;border-radius:10px;padding:8px 10px;text-align:center'>
        <div style='font-size:18px;font-weight:800;color:#e65100'>{len(caution_list)}</div>
        <div style='font-size:10px;color:#e65100;font-weight:600;margin-top:2px'>⚠️ 要注意</div>
      </div>
      <div style='flex:1;min-width:70px;background:#e8f5e9;border-radius:10px;padding:8px 10px;text-align:center'>
        <div style='font-size:18px;font-weight:800;color:#2e7d32'>{len(hold_list)}</div>
        <div style='font-size:10px;color:#2e7d32;font-weight:600;margin-top:2px'>✅ 保有継続</div>
      </div>
      <div style='flex:1;min-width:70px;background:#e8f5e9;border-radius:10px;padding:8px 10px;text-align:center'>
        <div style='font-size:18px;font-weight:800;color:#1b5e20'>{len(add_list)}</div>
        <div style='font-size:10px;color:#1b5e20;font-weight:600;margin-top:2px'>🎯 追加買い</div>
      </div>
    </div>
  </div>

  <!-- アラート -->
  <div style='padding:12px 12px 0'>{alert_html}</div>

  <!-- 銘柄カード一覧 -->
  <div style='padding:0 12px 24px'>
    <div style='font-size:11px;font-weight:700;color:#9e9e9e;letter-spacing:1px;
                text-transform:uppercase;margin:14px 0 10px'>銘柄別判断</div>
    {cards_html}
  </div>

  <!-- フッター -->
  <div style='text-align:center;font-size:10px;color:#bdbdbd;padding:0 12px 20px;line-height:1.6'>
    本レポートは参考情報です。投資判断はご自身の責任で行ってください。<br>
    生成: {datetime.now().strftime('%Y/%m/%d %H:%M')} | AI投資判断システム v2.0
  </div>

</div>
</body>
</html>"""

display(HTML(html_out))